# ESM C (Cambrian) Embeddings

![ESM C (Cambrian) Embeddings](https://proto-bio.github.io/proto-assets/images/tool/esmc/hero.png)

ESM C is an embedding-focused masked protein language model from EvolutionaryScale. Unlike ESM3, ESM C is purpose-built for producing high-quality contextualized embeddings: there is no sample/score interface.

Two open-weights variants are exposed here: `esmc_300m` (Cambrian Open License — commercial use OK) and `esmc_600m` (Cambrian Non-Commercial License — research/internal use only).

In [1]:
from proto_tools.utils.notebook_docs import (
    display_overview, display_api_reference, display_doc_link, display_available_tools,
)
display_doc_link("esmc")
display_overview("esmc")

# ESM C (Cambrian)

ESM C ("Cambrian") is EvolutionaryScale's embedding-focused protein language model. This toolkit wraps the openly licensed `esmc_300m` and `esmc_600m` models to produce per-sequence embeddings and optional per-position scores (logits) from supplied protein sequences. It provides only an embedding interface; it does not support sequence sampling or scoring.

## Available tools

In [2]:
display_available_tools("esmc")

- **`run_esmc_embeddings()`** — Extract protein sequence embeddings and logits using ESM C (Cambrian)

### `run_esmc_embeddings`

Returns a `SequenceEmbedding` per input sequence containing a mean-pooled embedding, the attention mask, and (optionally) per-position amino-acid logits.

In [3]:
import numpy as np

from proto_tools.tools.masked_models.esmc import (
    ESMCEmbeddingsConfig, ESMCEmbeddingsInput, run_esmc_embeddings,
)

# Two hemoglobin chains (alpha, beta) are close homologs; GFP is an unrelated control.
hba = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"
hbb = "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH"
gfp = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"

result = run_esmc_embeddings(ESMCEmbeddingsInput(sequences=[hba, hbb, gfp]), ESMCEmbeddingsConfig())
embs = [np.array(r.mean_embedding) for r in result.results]

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"Embedding dim: {embs[0].shape[0]}")
print(f"cosine(HBA, HBB) = {cosine(embs[0], embs[1]):.3f}  (homologous globins)")
print(f"cosine(HBA, GFP) = {cosine(embs[0], embs[2]):.3f}  (unrelated)")

Running run_esmc_embeddings [00:00]

Embedding dim: 960
cosine(HBA, HBB) = 0.971  (homologous globins)
cosine(HBA, GFP) = 0.057  (unrelated)


## API reference

In [4]:
display_api_reference("esmc", "input", "run_esmc_embeddings")
display_api_reference("esmc", "config", "run_esmc_embeddings")
display_api_reference("esmc", "output", "run_esmc_embeddings")

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Protein sequence(s) to process as string or list of strings. (will be normalized to List[str]) |

**Config** — `ESMCEmbeddingsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| <code>model_checkpoint</code> | <code>Literal['esmc_300m', 'esmc_600m']</code> | <code>'esmc_300m'</code> | ESM C weights variant ('esmc_300m' open license, 'esmc_600m' non-commercial) |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include per-position logits in the output (large; disable to save memory) |
| <code>repr_layer</code> | <code>int</code> | <code>-1</code> | Transformer layer index for embeddings; -1 returns post-norm output, others select pre-norm |

**Output** — `ESMCEmbeddingsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[SequenceEmbedding]</code> | required | Per-sequence embedding results |

**`SequenceEmbedding`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>mean_embedding</code> | <code>list[float]</code> | required | Mean-pooled embedding vector (averaged over sequence length) |
| <code>attention_mask</code> | <code>list[int]</code> | required | Binary mask: 1 = valid position, 0 = padding |
| <code>logits</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Per-position amino acid logits (seq_len, vocab_size) |
| <code>projection</code> | <code>Projection2D &#124; None</code> | <code>None</code> | 2D UMAP projection of this sequence's embedding within the call's batch |

**`Projection2D`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>x</code> | <code>float</code> | required | First reduced coordinate |
| <code>y</code> | <code>float</code> | required | Second reduced coordinate |